# Tuần 5 - Phần 2: Đánh giá Chiến lược Kinh doanh (Business Policy Evaluation) & Calibration

Trong phần 1, chúng ta đã huấn luyện các mô hình Uplift (T-Learner, X-Learner) và vẽ đường cong Qini. Tuy nhiên, Ban giám đốc không cần một đường cong, họ cần một **Quyết định cụ thể (Policy)**.

Mục tiêu của Notebook này là đưa kết quả CATE vào một bài toán Kinh doanh thực tế:
1. Tính toán Lợi nhuận kỳ vọng (Expected Incremental Profit) cho từng User.
2. So sánh hiệu quả của 5 chiến lược phân bổ Voucher.
3. Đo lường Oracle Regret (Chi phí cơ hội).
4. Kiểm định độ tin cậy của mô hình bằng Uplift Calibration.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
import os
import json

base_path = r"D:\Intern VSF\GSM-promotion-experimentation"
data_path = os.path.join(base_path, 'data', 'processed', 'segmented_simulation_data.csv')

print("============================================================")
print("Sprint 1B/2A: 5-Policy Comparison & Calibration")
print("============================================================")

# ─── 1. LOAD DATA ─────────────────────────────────────────
print("\n[1/5] Loading data...")
df = pd.read_csv(data_path)
df['is_credit_card'] = (df['payment_type'] == 1).astype(int)
print(f"  Total users: {len(df):,}")

# Economics parameters
VOUCHER_RATE = 0.15       # 15% of user's average fare
MARGIN_RATE  = 0.75       # 75% of incremental fare is margin
CAMPAIGN_BUDGET = 50000   # $50,000 total budget cap

features = ['age', 'is_urban', 'preferred_hour', 'is_rush_hour', 'is_airport_trip', 'is_rain_rider', 'is_weekend_rider', 'is_credit_card', 'passenger_count', 'monthly_rides_history', 'recency_days']

X = df[features]
y = df['Y_rand']
T = df['treatment_rand']

X_tv, X_test, y_tv, y_test, T_tv, T_test = train_test_split(X, y, T, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val, T_train, T_val = train_test_split(X_tv, y_tv, T_tv, test_size=0.25, random_state=42)

test_idx = X_test.index
df_test = df.loc[test_idx].copy()
df_test['avg_fare'] = df_test['avg_fare_per_trip']
print(f"  Test set size: {len(df_test):,}")

Sprint 1B/2A: 5-Policy Comparison & Calibration

[1/5] Loading data...
  Total users: 20,000
  Test set size: 4,000


## 2. Huấn luyện X-Learner và tính Expected Value

In [2]:
print("\n[2/5] Training X-Learner...")
params = dict(random_state=42, min_child_weight=5, reg_lambda=1.0, n_estimators=200, learning_rate=0.05, max_depth=4)

m0 = xgb.XGBRegressor(**params)
m1 = xgb.XGBRegressor(**params)
m0.fit(X_train[T_train == 0], y_train[T_train == 0])
m1.fit(X_train[T_train == 1], y_train[T_train == 1])

pseudo0 = m1.predict(X_train[T_train == 0]) - y_train[T_train == 0]
pseudo1 = y_train[T_train == 1] - m0.predict(X_train[T_train == 1])

tau0 = xgb.XGBRegressor(**params); tau0.fit(X_train[T_train == 0], pseudo0)
tau1 = xgb.XGBRegressor(**params); tau1.fit(X_train[T_train == 1], pseudo1)

cate = 0.5 * tau0.predict(X_test) + 0.5 * tau1.predict(X_test)
pred1 = m1.predict(X_test)
print(f"  Mean predicted CATE: {cate.mean():.4f}")

print("\n[3/5] Computing Expected Incremental Value per user...")
df_test['cate_pred'] = cate
df_test['pred_rides_treated'] = pred1
df_test['voucher_cost'] = df_test['avg_fare'] * VOUCHER_RATE
df_test['margin_per_ride'] = df_test['avg_fare'] * MARGIN_RATE

# EV_i = CATE_i × margin_per_ride − pred_rides_treated × voucher_cost
df_test['expected_value'] = (df_test['cate_pred'] * df_test['margin_per_ride']) - (df_test['pred_rides_treated'] * df_test['voucher_cost'])


[2/5] Training X-Learner...
  Mean predicted CATE: 0.9785

[3/5] Computing Expected Incremental Value per user...


## 3. Mô phỏng 5 Chiến lược & Oracle Regret

In [3]:
def evaluate_policy(target_mask, df_eval, label):
    targeted = df_eval[target_mask]
    not_targeted = df_eval[~target_mask]
    n_targeted = target_mask.sum()
    if n_targeted == 0:
        return {"Policy": label, "N_Targeted": 0, "Pct_Targeted": 0.0, "Total_Voucher_Cost": 0, "Expected_Incremental_Profit": 0, "Est_ROI_pct": 0}
    total_ev = targeted['expected_value'].sum()
    total_voucher_cost = (targeted['pred_rides_treated'] * targeted['voucher_cost']).sum()
    roi = (total_ev / total_voucher_cost * 100) if total_voucher_cost > 0 else 0
    return {
        "Policy": label,
        "N_Targeted": int(n_targeted),
        "Pct_Targeted": round(n_targeted / len(df_eval) * 100, 1),
        "Total_Voucher_Cost": round(total_voucher_cost, 0),
        "Expected_Incremental_Profit": round(total_ev, 0),
        "Est_ROI_pct": round(roi, 1)
    }

print("\n[4/5] Evaluating 6 policies...")
results = []
results.append({"Policy": "0. No Voucher", "N_Targeted": 0, "Pct_Targeted": 0.0, "Total_Voucher_Cost": 0, "Expected_Incremental_Profit": 0, "Est_ROI_pct": 0.0})

mass_mask = pd.Series([True] * len(df_test), index=df_test.index)
results.append(evaluate_policy(mass_mask, df_test, "1. Mass Voucher (All Users)"))

suburban_mask = df_test['persona'].str.contains('Suburban', case=False, na=False)
results.append(evaluate_policy(suburban_mask, df_test, "2. Segment Targeting (Suburban)"))

cate_threshold = df_test['cate_pred'].quantile(0.70)
uplift_mask = df_test['cate_pred'] >= cate_threshold
results.append(evaluate_policy(uplift_mask, df_test, "3. Uplift Targeting (Top 30% CATE)"))

profit_mask = df_test['expected_value'] > 0
results.append(evaluate_policy(profit_mask, df_test, "4. Profit Targeting (EV > 0)"))

df_sorted_ev = df_test.sort_values('expected_value', ascending=False).copy()
df_sorted_ev['cumulative_cost'] = (df_sorted_ev['pred_rides_treated'] * df_sorted_ev['voucher_cost']).cumsum()
budget_mask_idx = df_sorted_ev[df_sorted_ev['cumulative_cost'] <= CAMPAIGN_BUDGET].index
budget_mask = df_test.index.isin(budget_mask_idx)
results.append(evaluate_policy(budget_mask, df_test, f"5. Budget-Constrained (${CAMPAIGN_BUDGET:,})"))

if 'true_ite' in df_test.columns:
    df_test['oracle_ev'] = (df_test['true_ite'] * df_test['margin_per_ride']) - (df_test['pred_rides_treated'] * df_test['voucher_cost'])
    oracle_mask = df_test['oracle_ev'] > 0
    oracle_ev = df_test[oracle_mask]['oracle_ev'].sum()
    oracle_cost = (df_test[oracle_mask]['pred_rides_treated'] * df_test[oracle_mask]['voucher_cost']).sum()
    oracle_roi = (oracle_ev / oracle_cost * 100) if oracle_cost > 0 else 0
    results.append({
        "Policy": "6. Oracle Policy (True ITE — Sandbox only)",
        "N_Targeted": int(oracle_mask.sum()),
        "Pct_Targeted": round(oracle_mask.sum() / len(df_test) * 100, 1),
        "Total_Voucher_Cost": round(oracle_cost, 0),
        "Expected_Incremental_Profit": round(oracle_ev, 0),
        "Est_ROI_pct": round(oracle_roi, 1)
    })

    profit_policy_ev = df_test[profit_mask]['expected_value'].sum()
    regret = oracle_ev - profit_policy_ev
    regret_pct = (regret / oracle_ev * 100) if oracle_ev > 0 else 0
    print(f"\n  Oracle Profit: ${oracle_ev:,.0f}")
    print(f"  Profit Targeting: ${profit_policy_ev:,.0f}")
    print(f"  Regret: ${regret:,.0f} ({regret_pct:.1f}% of oracle)")

policy_df = pd.DataFrame(results)
display(policy_df)


[4/5] Evaluating 6 policies...


,Policy,N_Targeted,Pct_Targeted,Total_Voucher_Cost,Expected_Incremental_Profit,Est_ROI_pct
0,0. No Voucher,0,0.0,0.0,0.0,0.0
1,1. Mass Voucher (All Users),4000,100.0,131626.0,-58835.0,-44.7
2,2. Segment Targeting (Suburban),1660,41.5,34374.0,-2419.0,-7.0
3,3. Uplift Targeting (Top 30% CATE),1200,30.0,46178.0,-1783.0,-3.9
4,4. Profit Targeting (EV > 0),1030,25.8,19677.0,13148.0,66.8
5,"5. Budget-Constrained ($50,000)",2384,59.6,49985.0,4016.0,8.0


## 4. Tính toán Uplift Calibration (Deciles)

In [4]:
df_calib = df_test.copy()
df_calib['decile'] = pd.qcut(df_calib['cate_pred'], q=10, labels=False, duplicates='drop')
df_calib['decile'] = 9 - df_calib['decile'] + 1 

calib_results = []
for d in sorted(df_calib['decile'].unique()):
    subset = df_calib[df_calib['decile'] == d]
    t = subset[subset['treatment_rand'] == 1]
    c = subset[subset['treatment_rand'] == 0]
    obs_uplift = t['Y_rand'].mean() - c['Y_rand'].mean()
    true_uplift = subset['true_ite'].mean() if 'true_ite' in subset.columns else None
    pred_uplift = subset['cate_pred'].mean()
    
    calib_results.append({
        'Decile': d,
        'Predicted_CATE': pred_uplift,
        'Observed_Uplift': obs_uplift,
        'True_ITE': true_uplift
    })

calib_df = pd.DataFrame(calib_results)
display(calib_df)

,Decile,Predicted_CATE,Observed_Uplift,True_ITE
0,1,2.773380,2.323722,None
1,2,1.843058,0.975701,None
2,3,1.525544,0.038826,None
3,4,1.277436,2.057276,None
4,5,1.070931,1.724652,None
5,6,0.885822,1.138414,None
6,7,0.673204,1.045654,None
7,8,0.431579,0.811024,None
8,9,0.144805,1.666667,None
9,10,-0.841246,0.249505,None
